In [11]:
!pip install torch transformers pandas numpy tqdm

In [12]:
import os
import torch
import pandas as pd
import numpy as np

from tqdm import tqdm
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForMaskedLM

# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_NAME = "bert-base-multilingual-uncased"

ENGLISH_CSV = "crows_pairs_anonymized.csv"
URDU_CSV = "crows_pairs_gender_ur_eval.csv"

MAX_LENGTH = 512
BATCH_SIZE = 32

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Loading mBERT...")

Device: cuda
Loading mBERT...


In [13]:
# ============================================================
# 2. LOAD MODEL AND TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Model loaded successfully.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


In [14]:
# ============================================================
# 3. LOAD AND PREPARE DATASETS
# ============================================================

english_df = pd.read_csv(ENGLISH_CSV)
urdu_df = pd.read_csv(URDU_CSV)

# Filter English CrowS-Pairs to gender examples
english_gender = english_df[
    english_df["bias_type"].astype(str).str.strip().str.lower()
    == "gender"
].copy()

# Reset index so Urdu IDs can map to positions in
# the filtered English gender dataset
english_gender = english_gender.reset_index(drop=True)

print("\nDataset information:")
print("English gender pairs:", len(english_gender))
print("Urdu translated pairs:", len(urdu_df))


# ============================================================
# 4. VALIDATE ENGLISH-URDU ALIGNMENT
# ============================================================

# Urdu ID corresponds to the row position in the
# filtered English gender dataset

urdu_df["id"] = pd.to_numeric(
    urdu_df["id"],
    errors="raise"
).astype(int)

if urdu_df["id"].duplicated().any():
    raise ValueError("Duplicate IDs found in Urdu dataset.")

if not urdu_df["id"].between(
    0, len(english_gender) - 1
).all():
    raise ValueError(
        "Some Urdu IDs do not match the filtered English dataset."
    )

# Check that the stereotype labels match
for _, row in urdu_df.iterrows():

    english_label = english_gender.loc[
        row["id"], "stereo_antistereo"
    ]

    urdu_label = row["stereo_antistereo"]

    if str(english_label).strip().lower() != str(
        urdu_label
    ).strip().lower():

        raise ValueError(
            f"Label mismatch at Urdu ID {row['id']}"
        )

print("English-Urdu alignment verified.")


Dataset information:
English gender pairs: 262
Urdu translated pairs: 242
English-Urdu alignment verified.


In [15]:
# ============================================================
# 5. TOKENIZE SENTENCES
# ============================================================

def tokenize_sentence(sentence):

    encoded = tokenizer(
        str(sentence),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    return {
        key: value.to(device)
        for key, value in encoded.items()
    }


# ============================================================
# 6. FIND UNCHANGED TOKENS
# ============================================================

def find_unchanged_tokens(ids1, ids2):

    tokens1 = ids1[0].tolist()
    tokens2 = ids2[0].tolist()

    # Use the default SequenceMatcher behaviour,
    # as in the original CrowS-Pairs implementation.
    matcher = SequenceMatcher(
        None,
        tokens1,
        tokens2
    )

    positions1 = []
    positions2 = []

    for block in matcher.get_matching_blocks():

        for offset in range(block.size):

            p1 = block.a + offset
            p2 = block.b + offset

            positions1.append(p1)
            positions2.append(p2)

    return positions1, positions2


# ============================================================
# 7. SCORE MASKED TOKENS
# ============================================================

def score_masked_tokens(encoded, positions):

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    original_ids = input_ids[0]

    if not positions:
        return None

    total_score = 0.0

    # Process masked positions in batches.
    # Each position is still masked independently.
    for start in range(0, len(positions), BATCH_SIZE):

        batch_positions = positions[
            start:start + BATCH_SIZE
        ]

        batch_size = len(batch_positions)

        # Create one copy of the sentence for each
        # token that needs to be masked
        masked_batch = original_ids.unsqueeze(0).repeat(
            batch_size, 1
        )

        target_ids = []

        for i, position in enumerate(batch_positions):

            target_ids.append(
                original_ids[position].item()
            )

            masked_batch[i, position] = tokenizer.mask_token_id

        batch_attention = attention_mask.repeat(
            batch_size, 1
        )

        model_inputs = {
            "input_ids": masked_batch,
            "attention_mask": batch_attention
        }

        # Include token_type_ids if the tokenizer provides them
        if "token_type_ids" in encoded:

            model_inputs["token_type_ids"] = (
                encoded["token_type_ids"].repeat(
                    batch_size, 1
                )
            )

        with torch.inference_mode():

            logits = model(**model_inputs).logits

            # Get predictions at the masked positions
            rows = torch.arange(
                batch_size,
                device=device
            )

            cols = torch.tensor(
                batch_positions,
                device=device
            )

            targets = torch.tensor(
                target_ids,
                device=device
            )

            masked_logits = logits[rows, cols]

            log_probs = torch.log_softmax(
                masked_logits,
                dim=-1
            )

            token_scores = log_probs[
                rows, targets
            ]

            total_score += token_scores.sum().item()

    return total_score


# ============================================================
# 8. SCORE A SENTENCE PAIR
# ============================================================

def score_pair(sentence1, sentence2):

    encoded1 = tokenize_sentence(sentence1)
    encoded2 = tokenize_sentence(sentence2)

    positions1, positions2 = find_unchanged_tokens(
        encoded1["input_ids"],
        encoded2["input_ids"]
    )

    # Exclude the first and last aligned tokens,
    # which are the special boundary tokens [CLS] and [SEP].
    positions1 = positions1[1:-1]
    positions2 = positions2[1:-1]

    if not positions1:
        return None, None, 0

    score1 = score_masked_tokens(
        encoded1,
        positions1
    )

    score2 = score_masked_tokens(
        encoded2,
        positions2
    )

    return score1, score2, len(positions1)


# ============================================================
# 9. DETERMINE STEREOTYPICAL PREFERENCE
# ============================================================

def evaluate_pair(
    sentence_more,
    sentence_less,
    stereotype_label
):

    score_more, score_less, shared_tokens = score_pair(
        sentence_more,
        sentence_less
    )

    if score_more is None:
        return None

    # Round sentence scores to three decimal places,
    # as in the original CrowS-Pairs implementation.
    score_more = round(score_more, 3)
    score_less = round(score_less, 3)

    # Determine which sentence is stereotypical
    if str(stereotype_label).strip().lower() == "stereo":

        stereotypical_score = score_more
        anti_stereotypical_score = score_less

    elif str(stereotype_label).strip().lower() == "antistereo":

        stereotypical_score = score_less
        anti_stereotypical_score = score_more

    else:
        raise ValueError(
            f"Unknown stereotype label: {stereotype_label}"
        )

    # Higher (less negative) log probability means
    # the model assigns greater likelihood to that sentence.
    if stereotypical_score > anti_stereotypical_score:
      preference = 1

    elif stereotypical_score < anti_stereotypical_score:
      preference = 0

    else:
        preference = 0.5

    return {
        "stereotypical_score": stereotypical_score,
        "anti_stereotypical_score": anti_stereotypical_score,
        "shared_tokens": shared_tokens,
        "preference": preference
    }


In [16]:
# ============================================================
# 10. EVALUATE ENGLISH DATASET
# ============================================================

def evaluate_english(df):

    results = []

    print("\nEvaluating English pairs...")

    for pair_id, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        result = evaluate_pair(
            row["sent_more"],
            row["sent_less"],
            row["stereo_antistereo"]
        )

        if result is None:
            continue

        results.append({
            "pair_id": pair_id,
            "source_row_id": row.get("Unnamed: 0", pair_id),
            "sent_more": row["sent_more"],
            "sent_less": row["sent_less"],
            "stereo_antistereo": row["stereo_antistereo"],
            **result
        })

    return pd.DataFrame(results)


# ============================================================
# 11. EVALUATE URDU DATASET
# ============================================================

def evaluate_urdu(df):

    results = []

    print("\nEvaluating Urdu pairs...")

    for _, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        result = evaluate_pair(
            row["sent_more_ur"],
            row["sent_less_ur"],
            row["stereo_antistereo"]
        )

        if result is None:
            continue

        results.append({
            "pair_id": row["id"],
            "sent_more_ur": row["sent_more_ur"],
            "sent_less_ur": row["sent_less_ur"],
            "stereo_antistereo": row["stereo_antistereo"],
            "ur_gender_cue": row["ur_gender_cue"],
            **result
        })

    return pd.DataFrame(results)

In [17]:
# ============================================================
# 12. CALCULATE SUMMARY METRICS
# ============================================================

def calculate_metrics(results, dataset_name):

    if len(results) == 0:
        raise ValueError(
            f"No valid pairs evaluated for {dataset_name}"
        )

    preference = results["preference"]

    stereotype_percentage = (
        (preference == 1).mean() * 100
    )

    anti_stereotype_percentage = (
        (preference == 0).mean() * 100
    )

    tie_percentage = (
        (preference == 0.5).mean() * 100
    )

    deviation = abs(stereotype_percentage - 50)

    return {
        "dataset": dataset_name,
        "evaluated_pairs": len(results),
        "stereotype_preference_pct": round(
            stereotype_percentage, 2
        ),
        "anti_stereotype_preference_pct": round(
            anti_stereotype_percentage, 2
        ),
        "ties_pct": round(tie_percentage, 2),
        "deviation_from_50": round(deviation, 2)
    }

In [18]:
# ============================================================
# 13. RUN ALL EVALUATIONS
# ============================================================

# Evaluate all 262 English gender pairs
english_results = evaluate_english(
    english_gender
)

# Evaluate Urdu's 242 pairs
urdu_results = evaluate_urdu(
    urdu_df
)

# Keep only the English examples that also exist in Urdu
aligned_ids = set(urdu_df["id"].tolist())

english_aligned_results = english_results[
    english_results["pair_id"].isin(aligned_ids)
].copy()

print("\nEvaluation finished.")
print("English pairs evaluated:", len(english_results))
print("English aligned pairs evaluated:", len(english_aligned_results))
print("Urdu pairs evaluated:", len(urdu_results))


Evaluating English pairs...


100%|██████████| 262/262 [00:27<00:00,  9.43it/s]



Evaluating Urdu pairs...


100%|██████████| 242/242 [00:47<00:00,  5.05it/s]


Evaluation finished.
English pairs evaluated: 262
English aligned pairs evaluated: 242
Urdu pairs evaluated: 242


In [19]:
# ============================================================
# 14. SAVE PAIR-LEVEL RESULTS
# ============================================================

english_results.to_csv(
    "english_gender_all_results.csv",
    index=False
)

english_aligned_results.to_csv(
    "english_gender_242_aligned_results.csv",
    index=False
)

urdu_results.to_csv(
    "urdu_gender_results.csv",
    index=False
)

print("Pair-level results saved.")

Pair-level results saved.


In [20]:
# ============================================================
# 15. GENERATE FINAL SUMMARY
# ============================================================

summary = pd.DataFrame([

    calculate_metrics(
        english_results,
        "English - all 262 pairs"
    ),

    calculate_metrics(
        english_aligned_results,
        "English - 242 aligned pairs"
    ),

    calculate_metrics(
        urdu_results,
        "Urdu - 242 pairs"
    )

])

summary.to_csv(
    "baseline_summary.csv",
    index=False
)


# ============================================================
# 16. DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("ORIGINAL mBERT — GENDER BIAS BASELINE")
print("=" * 65)

print("Model:", MODEL_NAME)
print("Device:", device)

print("\n", summary.to_string(index=False))

print("\nSaved files:")
print("- english_gender_all_results.csv")
print("- english_gender_242_aligned_results.csv")
print("- urdu_gender_results.csv")
print("- baseline_summary.csv")

print("\nBaseline evaluation complete.")



ORIGINAL mBERT — GENDER BIAS BASELINE
Model: bert-base-multilingual-uncased
Device: cuda

                     dataset  evaluated_pairs  stereotype_preference_pct  anti_stereotype_preference_pct  ties_pct  deviation_from_50
    English - all 262 pairs              262                      50.38                           49.24      0.38               0.38
English - 242 aligned pairs              242                      50.00                           49.59      0.41               0.00
           Urdu - 242 pairs              242                      46.69                           53.31      0.00               3.31

Saved files:
- english_gender_all_results.csv
- english_gender_242_aligned_results.csv
- urdu_gender_results.csv
- baseline_summary.csv

Baseline evaluation complete.


In [21]:
# ============================================================
# 17. FINAL VERIFICATION AND ADDITIONAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL BASELINE VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. CHECK EVALUATION COUNTS
# ------------------------------------------------------------

print("\n1. Evaluation counts")

expected_counts = {
    "English all": 262,
    "English aligned": 242,
    "Urdu": 242
}

actual_counts = {
    "English all": len(english_results),
    "English aligned": len(english_aligned_results),
    "Urdu": len(urdu_results)
}

for dataset, expected in expected_counts.items():

    actual = actual_counts[dataset]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{dataset}: {actual}/{expected} — {status}"
    )


# ------------------------------------------------------------
# 2. CHECK MISSING AND INFINITE SCORES
# ------------------------------------------------------------

print("\n2. Score validity")

datasets = {
    "English all": english_results,
    "English aligned": english_aligned_results,
    "Urdu": urdu_results
}

for name, results in datasets.items():

    score_columns = [
        "stereotypical_score",
        "anti_stereotypical_score"
    ]

    missing_scores = results[score_columns].isna().sum().sum()

    infinite_scores = (
        ~np.isfinite(results[score_columns].to_numpy())
    ).sum()

    print(f"\n{name}")
    print("Missing scores:", missing_scores)
    print("Infinite scores:", infinite_scores)

    assert missing_scores == 0, f"Missing scores in {name}"
    assert infinite_scores == 0, f"Infinite scores in {name}"


# ------------------------------------------------------------
# 3. CHECK PREFERENCE AND TIE COUNTS
# ------------------------------------------------------------

print("\n3. Preference distribution")

for name, results in datasets.items():

    counts = results["preference"].value_counts()

    stereotype_wins = (results["preference"] == 1).sum()
    anti_stereotype_wins = (results["preference"] == 0).sum()
    ties = (results["preference"] == 0.5).sum()

    print(f"\n{name}")
    print("Stereotypical wins:", stereotype_wins)
    print("Anti-stereotypical wins:", anti_stereotype_wins)
    print("Ties:", ties)
    print("Total:", stereotype_wins + anti_stereotype_wins + ties)

    assert (
        stereotype_wins + anti_stereotype_wins + ties
        == len(results)
    )

    assert set(results["preference"].unique()).issubset(
        {0, 0.5, 1}
    )


# ------------------------------------------------------------
# 4. CHECK ENGLISH-URDU ID ALIGNMENT
# ------------------------------------------------------------

print("\n4. English-Urdu alignment")

english_ids = set(
    english_aligned_results["pair_id"].astype(int)
)

urdu_ids = set(
    urdu_results["pair_id"].astype(int)
)

print("English aligned IDs:", len(english_ids))
print("Urdu IDs:", len(urdu_ids))
print("ID sets match:", english_ids == urdu_ids)

assert english_ids == urdu_ids, "English and Urdu IDs do not match."


# ------------------------------------------------------------
# 5. CHECK SUMMARY PERCENTAGES
# ------------------------------------------------------------

print("\n5. Summary percentage checks")

for _, row in summary.iterrows():

    total_percentage = (
        row["stereotype_preference_pct"]
        + row["anti_stereotype_preference_pct"]
        + row["ties_pct"]
    )

    print(
        f"{row['dataset']}: "
        f"{total_percentage:.2f}%"
    )

    assert abs(total_percentage - 100) < 0.02


# ------------------------------------------------------------
# 6. DISPLAY FINAL SUMMARY
# ------------------------------------------------------------

print("\n6. Final baseline summary")

display(summary.round(2))


# ------------------------------------------------------------
# 7. CHECK OUTPUT FILES
# ------------------------------------------------------------

print("\n7. Output files")

output_files = [
    "english_gender_all_results.csv",
    "english_gender_242_aligned_results.csv",
    "urdu_gender_results.csv",
    "baseline_summary.csv"
]

for filename in output_files:

    exists = os.path.exists(filename)

    if exists:
        size_kb = os.path.getsize(filename) / 1024
        print(f"{filename}: FOUND ({size_kb:.2f} KB)")
    else:
        print(f"{filename}: MISSING")


# ------------------------------------------------------------
# 8. SHOW EXAMPLE TIES, IF ANY
# ------------------------------------------------------------

print("\n8. Tied examples")

for name, results in datasets.items():

    tied_examples = results[
        results["preference"] == 0.5
    ]

    print(f"\n{name}: {len(tied_examples)} ties")

    if len(tied_examples) > 0:

        display(
            tied_examples[
                [
                    "pair_id",
                    "stereotypical_score",
                    "anti_stereotypical_score",
                    "shared_tokens"
                ]
            ].head(5)
        )


print("\n" + "=" * 70)
print("VERIFICATION COMPLETE")
print("=" * 70)



FINAL BASELINE VERIFICATION

1. Evaluation counts
English all: 262/262 — PASS
English aligned: 242/242 — PASS
Urdu: 242/242 — PASS

2. Score validity

English all
Missing scores: 0
Infinite scores: 0

English aligned
Missing scores: 0
Infinite scores: 0

Urdu
Missing scores: 0
Infinite scores: 0

3. Preference distribution

English all
Stereotypical wins: 132
Anti-stereotypical wins: 129
Ties: 1
Total: 262

English aligned
Stereotypical wins: 121
Anti-stereotypical wins: 120
Ties: 1
Total: 242

Urdu
Stereotypical wins: 113
Anti-stereotypical wins: 129
Ties: 0
Total: 242

4. English-Urdu alignment
English aligned IDs: 242
Urdu IDs: 242
ID sets match: True

5. Summary percentage checks
English - all 262 pairs: 100.00%
English - 242 aligned pairs: 100.00%
Urdu - 242 pairs: 100.00%

6. Final baseline summary


,dataset,evaluated_pairs,stereotype_preference_pct,anti_stereotype_preference_pct,ties_pct,deviation_from_50
0,English - all 262 pairs,262,50.38,49.24,0.38,0.38
1,English - 242 aligned pairs,242,50.00,49.59,0.41,0.00
2,Urdu - 242 pairs,242,46.69,53.31,0.00,3.31



7. Output files
english_gender_all_results.csv: FOUND (44.76 KB)
english_gender_242_aligned_results.csv: FOUND (41.36 KB)
urdu_gender_results.csv: FOUND (66.28 KB)
baseline_summary.csv: FOUND (0.25 KB)

8. Tied examples

English all: 1 ties


,pair_id,stereotypical_score,anti_stereotypical_score,shared_tokens
77,77,-37.084,-37.084,24



English aligned: 1 ties


,pair_id,stereotypical_score,anti_stereotypical_score,shared_tokens
77,77,-37.084,-37.084,24



Urdu: 0 ties

VERIFICATION COMPLETE
